# VCC 2026 — lavoro remoto su cellule singole (Colab + Drive)

**Cosa fare:** esegui la **cella 1** (autorizza Drive quando te lo chiede), poi la **cella 2** e lascia
la scheda aperta. La cella 2 resta in esecuzione (è normale: così Colab non chiude la sessione per
inattività) e stampa una riga per ogni job che parte o finisce.

Il lead deposita i job in `MyDrive/vcc2026/runs/queue/` da Drive per desktop; il dispatcher li esegue
in background (al massimo due insieme) e scrive i log in `MyDrive/vcc2026/runs/jobs/`.
Per fermare tutto: pulsante ■ sulla cella 2.

Il notebook non contiene logica scientifica: i job sono file versionati in `notebooks/colab_jobs/`
e il codice è la copia di `MyDrive/vcc2026/code`, ricopiata prima di ogni job.

In [ ]:
# Cella 1 — setup
import os, sys, shutil, subprocess, time, threading, json
from pathlib import Path
if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/vcc2026')
CODE = Path('/content/vcc')
QUEUE = DRIVE / 'runs' / 'queue'
JOBLOG = DRIVE / 'runs' / 'jobs'
for d in (QUEUE, JOBLOG):
    d.mkdir(parents=True, exist_ok=True)
os.environ['VCC2026_DATA_ROOT'] = str(DRIVE / 'data')

def sync_code():
    tmp = Path('/content/vcc_new')
    if tmp.exists():
        shutil.rmtree(tmp)
    shutil.copytree(DRIVE / 'code', tmp, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
    if CODE.exists():
        shutil.rmtree(CODE)
    tmp.rename(CODE)
    stamp = CODE / 'SYNC_STAMP.txt'
    return stamp.read_text().strip() if stamp.exists() else 'no stamp'

print('code:', sync_code())
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cell-eval2==0.16.0', 'scanpy', 'polars', 'pyarrow'],
                   capture_output=True, text=True)
print('pip rc', r.returncode, r.stderr[-1500:])
subprocess.run('free -g; df -h /content | tail -1; nproc; python -V; nvidia-smi -L 2>/dev/null || echo no-gpu', shell=True)

In [ ]:
# Cella 2 — dispatcher: resta in esecuzione (■ per fermarlo)
MAX_PARALLEL = 2
running, seen = {}, set()

def note(msg):
    line = f"{time.strftime('%Y-%m-%dT%H:%M:%S')} {msg}"
    print(line, flush=True)
    with open(JOBLOG / 'dispatcher.log', 'a') as fh:
        print(line, file=fh)

note('dispatcher up; queue = ' + str(QUEUE))
last_beat = time.time()
while True:
    try:
        for name, (proc, logf) in list(running.items()):
            rc = proc.poll()
            if rc is not None:
                (QUEUE / f'{name}.done').write_text(json.dumps({'rc': rc, 'finished': time.strftime('%Y-%m-%dT%H:%M:%S')}))
                note(f'finished {name} rc={rc}')
                del running[name]
        os.listdir(QUEUE)  # nudge the Drive mount to refresh the listing
        for job in sorted(QUEUE.glob('*.sh')):
            name = job.name
            if name in seen or (QUEUE / f'{name}.started').exists():
                seen.add(name)
                continue
            if len(running) >= MAX_PARALLEL:
                break
            seen.add(name)
            stamp = sync_code()
            local = Path('/content/jobs') / name
            local.parent.mkdir(exist_ok=True)
            shutil.copy(job, local)
            logf = JOBLOG / f"{name[:-3]}.log"
            proc = subprocess.Popen(['bash', str(local)], stdout=open(logf, 'w'), stderr=subprocess.STDOUT,
                                    env=os.environ.copy())
            (QUEUE / f'{name}.started').write_text(json.dumps({'pid': proc.pid, 'code': stamp,
                                                                 'started': time.strftime('%Y-%m-%dT%H:%M:%S')}))
            running[name] = (proc, logf)
            note(f'started {name} pid={proc.pid} code={stamp}')
        if time.time() - last_beat > 600:
            mem = subprocess.run('free -g | head -2 | tail -1', shell=True, capture_output=True, text=True).stdout.split()
            note(f"alive; running={list(running)}; RAM used/total GiB {mem[2] if len(mem) > 2 else '?'}/{mem[1] if len(mem) > 1 else '?'}")
            last_beat = time.time()
    except Exception as exc:  # keep the loop alive; the error is logged
        note(f'dispatcher error: {type(exc).__name__}: {exc}')
    time.sleep(20)

In [ ]:
# Cella 3 — stato (solo se la cella 2 è ferma)
subprocess.run(f'tail -n 20 {JOBLOG}/dispatcher.log; for f in $(ls -t {JOBLOG}/*.log | head -3); do echo == $f; tail -n 12 $f; done; '
               'free -g | head -2; df -h /content | tail -1', shell=True)